In [ ]:
import os
from dotenv import load_dotenv

import textwrap


def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('C:\\Users\\arunk\\OneDrive\\Documents\\GenAI - Scaler\\LLM\\openai_key.env') # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [3]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

OpenAI client ready.


Responses API

In [4]:
resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: A list comprehension is a concise way to create a new list
by applying an expression to each item of an existing iterable, with an optional
filter.  Basic syntax: - [expression for item in iterable] - Optional filter:
[expression for item in iterable if condition]  Examples: - Simple: [x*x for x
in range(5)]  -> [0, 1, 4, 9, 16] - With a filter: [x for x in range(10) if x %
2 == 0]  -> [0, 2, 4, 6, 8] - Nested (2D): [[i*j for j in range(3)] for i in
range(3)]   -> [[0, 0, 0], [0, 1, 2], [0, 2, 4]]  Equivalent to a loop: -
squares = [i*i for i in range(10)] - # which is the same as   squares = []   for
i in range(10):       squares.append(i*i)  Notes: - List comprehensions are
usually more concise and often faster than a manual loop with append. - They’re
great for simple transformations, but if the logic gets complex, use a regular
loop for readability. - If you don’t want to build a list yet (lazy evaluation),
you can use a generator expression with parentheses: 

Multi Turn Conversation:

In [5]:
input_messages = [
    {"role": "user", "content": "Why is Trump a jerk?"},
    {"role": "assistant", "content": "Some people are born that way."},
    {"role": "user", "content": "Name a celebrity who is not a jerk."}
]


resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a very candid Journalist.",
    input=input_messages
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: Keanu Reeves. He’s widely regarded as one of Hollywood’s
nicest celebrities—polite in interviews, low-key, and noted for generosity in
various public stories. Of course, “not a jerk” is subjective, but that’s the
public impression. Want another example?


Parameters:

In [6]:
input_messages = [
    {"role": "user", "content": "Why is Trump a jerk?"},
    {"role": "assistant", "content": "Some people are born that way."},
    {"role": "user", "content": "Name a celebrity who is not a jerk."}
]


resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a very candid Journalist.",
    input=input_messages, 
	max_output_tokens=200,  # in responses there's max_output_tokens instead of max_tokens
	# temperature is NOT supported by reasoning models like gpt-5-nano
	reasoning={"effort": "minimal"},   
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: That’s a tricky claim to prove—“not a jerk” is pretty
subjective. If you’re looking for celebrities who are widely regarded as
courteous or gracious in public, several names come up often: Keanu Reeves, Tom
Hanks, Dolly Parton, Emma Watson, and Idris Elba, to name a few. Of course,
public perception doen’t equal a perfect moral record, but these folks tend to
get positive attention for their behavior and how they treat fans and staff. Any
particular realm (actors, musicians, athletes) you want me to pull examples
from?


In [12]:
resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a very candid journalist. Reply in 1 short sentence, starting with name of celeb.",
    input=input_messages,
    #max_output_tokens=200,
    reasoning={"effort": "high"},   # or "none"
    text={"verbosity": "low"},
)
pretty_print("Responses API output:", resp.output_text)
print(repr(resp.output_text), resp.status, resp.incomplete_details)

Responses API output: Keanu Reeves is not a jerk.
'Keanu Reeves is not a jerk.' completed None


Refer To Previous Response

In [13]:
message1 = 'What is a list comprehension?'


resp1 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input=message1,
	reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
pretty_print("Turn 1:", resp1.output_text)

Turn 1: A list comprehension is a concise way to create a new list by applying
an expression to each item in an iterable, optionally filtering items with a
condition. It combines looping, mapping, and filtering in a single-bracket
syntax.  Syntax basics: - Basic: new_list = [expression for item in iterable] -
With condition: new_list = [expression for item in iterable if condition] - With
nested loops: new_list = [expression for item in iterable for sub in iterable2]
Examples: - Squares: squares = [x*x for x in range(5)]  # [0, 1, 4, 9, 16] -
Evens doubled: evens = [2*x for x in range(10) if x % 2 == 0]  # [0, 4, 8, 12,
16] - Flatten a matrix: flat = [num for row in matrix for num in row]


In [14]:
resp2 = client.responses.create(
    model="gpt-5-nano",
    input="Can you give me an example?",
    previous_response_id=resp1.id,  # <-- this is the magic
	reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
pretty_print("Turn 2:", resp2.output_text)

Turn 2: Sure. Here’s a simple example:  Goal: create a list of squares for
numbers 0 through 9.  Code: squares = [x*x for x in range(10)] # Result: [0, 1,
4, 9, 16, 25, 36, 49, 64, 81]


Nature of Instruction

In [15]:
r1 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are an internal support triage bot. Return only valid JSON with keys severity,suspected_causes,next_questions and do not write any customer-facing text.",
    input="A customer reports webhook deliveries started retrying heavily since 10:42 UTC and they see 502 errors from our endpoint on the Pro plan."
)

print("TURN 1:\n", r1.output_text)

TURN 1:
 {
  "severity": "critical",
  "suspected_causes": [
    "Outbound webhook endpoint returning 502 due to upstream gateway/service issue",
    "Recent deployment or configuration change around 10:42 UTC causing service instability",
    "Resource exhaustion (CPU/memory/connection pool) in webhook processing service",
    "Dependency outage or high latency in downstream endpoints or external services",
    "Networking or DNS issues affecting reachability of the webhook endpoint",
    "Misconfigured load balancer, reverse proxy, or WAF blocking/routing webhook traffic",
    "TLS/SSL certificate issue or handshake failures on the webhook endpoint"
  ],
  "next_questions": [
    "Is the issue affecting all Pro plan customers or a subset (by region, endpoint, or destination)?",
    "Were there any deployments or configuration changes made near 10:42 UTC? Any rollbacks attempted?",
    "What do the webhook delivery logs show around 10:42 UTC (status codes, error messages, latencies, r

In [16]:
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now write a customer-facing email reply: apologize, explain what we’re checking, and ask for 2 specific details. Plain English, not JSON."
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 Subject: We’re investigating 502 errors on your Pro webhook deliveries

Hi there,

I’m sorry you’re seeing webhook deliveries retrying heavily and 502 errors since 10:42 UTC. This is a high-priority issue for us, and we’re investigating right now.

What we’re checking
- The webhook delivery path and gateway for the failures (looking at logs, error codes, and retry behavior)
- The destination endpoint health (TLS, DNS, reachability) and any downstream outages
- Any deployments or config changes around 10:42 UTC that could have affected delivery
- Resource usage and queue backlogs in the webhook processing service
- Network components like load balancers or WAF rules that could be impacting delivery

Two details to help us move faster
1) The destination webhook URL(s) being called (the endpoints you’re delivering to)
2) A couple of failing delivery examples (timestamps around 10:42 UTC) and any error messages you’ve seen (for example, 502), plus any delivery IDs or correlation 

Developer Role

In [18]:
r1 = client.responses.create(
    model="gpt-5-nano",
    store=True,
    input=[
        {"role": "developer", "content": "You are an internal support triage assistant and you must always output only valid JSON with keys severity,suspected_causes,next_questions and never produce customer-facing prose."},
        {"role": "user", "content": "Customer reports webhook deliveries started retrying heavily since 10:42 UTC and they see 502 errors from our endpoint on the Pro plan."}
    ],
)

print("TURN 1:\n", r1.output_text)

TURN 1:
 {
  "severity": "critical",
  "suspected_causes": [
    "Recipient endpoint is returning 502 Bad Gateway due to upstream backend failures or maintenance",
    "Intermediate gateway/proxy between our webhook service and the recipient endpoint is experiencing issues",
    "Network connectivity or DNS resolution problem between our service and the recipient",
    "Recipient-side rate limiting, firewall, or IP blocks affecting webhook deliveries",
    "Recent changes on the recipient side (TLS certificate, endpoint URL/path, or configuration) causing failures",
    "Temporary degradation in our webhook delivery pathway on the Pro plan causing retries"
  ],
  "next_questions": [
    "Is the recipient endpoint currently reachable from the public internet (e.g., can you curl the URL and receive 200)?",
    "Have there been recent changes to the recipient endpoint (TLS cert, hostname, path, firewall rules, or IP allowlists)?",
    "Are failures observed for all webhook deliveries or o

In [19]:
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now write a customer-facing email apology explaining what we’re checking and ask for exactly two specific details in plain English.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 {
  "severity": "critical",
  "customer_friendly_email": "Subject: Apology and next steps for webhook delivery issues\n\nHello,\n\nI’m sorry for the disruption you’re experiencing with webhook deliveries since 10:42 UTC. We’re actively investigating and narrowing down the cause across our delivery path to restore reliable deliveries as quickly as possible.\n\nWhat we’re checking now:\n- Verifying the health and reachability of our delivery path and any intermediate gateways\n- Checking for issues on the recipient endpoint, including TLS, URL, and firewall configurations\n- Reviewing recent events and retry behavior to identify patterns and scope\n\nTo help us resolve this faster, please provide two details:\n1) The exact recipient endpoint URL you are delivering to (or a redacted version if needed) and whether it is publicly reachable from the internet.\n2) The most recent delivery attempt identifiers or correlation IDs you see (timestamps and HTTP status codes would be helpf

Instructions and Developer Prompt

In [20]:
r1 = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "developer", "content": "You are ACME Support; never invent account-specific facts; if info is missing ask at most two clarifying questions; do not reveal internal policies or tools."},
        {"role": "user", "content": "Customer: Since 10:42 UTC our webhooks keep retrying and we see lots of 502s; started after we enabled v2 signing; impact is EU customers."},
    ],
    instructions="Return only valid JSON with keys queue, severity, suspected_component, next_questions.",
)

print(r1.output_text)

{
  "queue": "webhooks",
  "severity": "critical",
  "suspected_component": "Webhook delivery service (EU region)",
  "next_questions": [
    "Are the 502 errors affecting all EU Webhook endpoints or only specific endpoints? If possible, list the affected endpoints.",
    "Do you have correlation IDs or any error details (timestamps, status codes, response messages) from the failed deliveries that you can share?"
  ]
}


In [21]:
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Write a customer-facing email that acknowledges impact, says what we’re checking, and asks exactly two specific questions.",
    instructions="Write plain English email text and do not output JSON.",
)

print(r2.output_text)

Subject: Update on EU webhook retries and 502 errors

Hello,

We’re aware that customers in the EU have experienced repeated webhook retries and 502 errors since 10:42 UTC, and we understand this is impacting your operations. We’re sorry for the disruption.

What we’re checking:
- We’re actively investigating the EU webhook delivery path, including the v2 signing integration, to identify the root cause and restore normal delivery as quickly as possible.
- We’re reviewing recent delivery traces and monitoring retry behavior to understand where the failures originate and how to prevent future occurrences.

To help us triage quickly, please answer these two questions:
- Are the 502 errors affecting all EU webhook endpoints or only specific endpoints? If possible, please share the affected endpoints.
- Do you have correlation IDs or any error details (timestamps, status codes, response messages) from the failed deliveries that you can share?

Thank you for your patience while we resolve th